# ⚡ 1교시: Airflow UI + 아키텍처 개요

## 🎯 학습 목표
- Airflow가 관리하는 범위(오케스트레이션)와 역할을 설명할 수 있다.
- Required components와 Optional components의 역할을 구분할 수 있다.
- GitHub Actions와 Airflow의 운영 관점 차이를 구분할 수 있다.
- 배포 아키텍처(Basic/Distributed/Separate DAG processing)의 트레이드오프를 설명할 수 있다.
- Airflow UI에서 DAG, DAG Run, Task Instance 관점을 분리해 해석할 수 있다.
- 장애 대응 시 `Home -> Dag List -> Grid -> Task Logs` 기본 동선을 실습할 수 있다.

---
## 0. 실습 준비

이 교시는 `Day29-airflow-beginner-lab/` 환경을 기준으로 UI를 확인합니다.

- 기준 이론: `01_ui_overview_handout.md`, `02_airflow_overview_handout.md`
- 실습 환경: Airflow 3.1.7 + Postgres + beginner DAG 2개

In [ ]:
# === Checkpoint 0-1: 환경 시작 ===
cd Day29-airflow-beginner-lab
cp -n .env.example .env

docker compose -f compose.yml up airflow-init

docker compose -f compose.yml up -d

docker compose -f compose.yml ps

---
## 1. Airflow는 무엇을 관리하는가

Airflow는 "데이터 처리 코드 자체"보다 "실행 순서/조건/재시도/관측"을 관리하는 오케스트레이션 플랫폼입니다.

핵심 포인트:
- DAG(Directed Acyclic Graph)로 의존성을 정의
- Scheduler가 실행 시점을 결정
- UI(Webserver/API Server)에서 운영자가 상태를 관측하고 재실행

### 1.1 많이 사용되는 영역
- 데이터 엔지니어링 파이프라인
- ETL/ELT 배치 작업
- ML 피처 생성/학습 파이프라인
- 리포트/정산/정기 집계 자동화

### 1.2 왜 Airflow가 필요한가

예: 매일 새벽 배치
1. 수집(extract)
2. 전처리(transform)
3. 적재(load)
4. 리포트 생성
5. 실패 시 알림

수동 운영으로는 의존성, 재시도, 스케줄, 실패 원인 추적을 안정적으로 유지하기 어렵습니다.
Airflow는 이 운영 부담을 DAG + 정책으로 표준화합니다.

### 1.3 구성요소

#### Required components
- Scheduler: 스케줄/의존성 기반으로 Task 실행 시점 결정
- DAG Processor: DAG 파일 파싱/직렬화
- API Server(Web 진입점): UI/API 요청 처리
- DAG files folder: 워크플로우 코드 저장 위치
- Metadata DB: 실행 상태/이력 저장

#### Optional components
- Triggerer: deferrable task 처리
- Worker: 분산 실행 환경에서 Task 실행
- Plugins folder: 연산자/훅 확장

### 1.4 GitHub Actions와 비교

| 관점 | Airflow | GitHub Actions |
|---|---|---|
| 중심 모델 | DAG 기반 오케스트레이션 | 리포지토리 이벤트 기반 자동화 |
| 강점 | 백필/재처리, 데이터 간격, 운영 UI | CI/CD, PR/배포 파이프라인 |
| 실행 주체 | Scheduler + Worker/Executor | GitHub-hosted/self-hosted Runner |
| 운영 초점 | 파이프라인 가시성/장애 대응 | 빌드/테스트/배포 자동화 |

정리:
- "백필/재처리 + 운영 UI"가 핵심이면 Airflow
- "코드 변경 이벤트 중심 자동화"가 핵심이면 GitHub Actions

---
## 2. 배포 아키텍처 패턴

공식 문서 기준 3단계 관점을 기억합니다.

### 2.1 Basic Airflow deployment
![Basic Airflow deployment](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/diagram_basic_airflow_architecture.png)

- 단일 머신 중심 구조
- 학습/PoC에는 단순하고 빠름
- 실무 확장 시 보안/격리/확장성 제약이 빠르게 드러남

### 2.2 Distributed Airflow architecture
![Distributed Airflow architecture](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/diagram_distributed_airflow_architecture.png)

- Scheduler/Webserver/Worker/DB를 분리해 배치
- 병렬성과 안정성을 높이기 쉬움
- 네트워크/권한/동기화 설계가 중요

### 2.3 Separate DAG processing architecture
![Separate DAG processing architecture](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/diagram_dag_processor_airflow_architecture.png)

- DAG 파싱을 별도 프로세스로 분리
- 스케줄러 가용성/보안 경계 강화
- 코드 실행 영역과 스케줄링 영역의 책임 분리

### 2.4 왜 DAG Processor를 분리하는가

분리 전(스케줄러 직접 파싱):
- 무거운 Top-level 코드가 스케줄러를 직접 블로킹 가능
- 스케줄러 권한 범위에서 민감정보 접근 리스크 증가

분리 후(DAG Processor 파싱 + 직렬화 전달):
- 위험 코드 영향 범위를 DAG Processor 측으로 제한
- 스케줄러는 Python 코드 대신 직렬화된 메타정보 중심으로 동작
- 장애 격리와 운영 복원력 향상

### 2.5 Workloads와 Control Flow 핵심
![DAGs visual](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/dags.png)
![Edge label example](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/edge_label_example.png)

- Task 유형: Operators, Sensors, TaskFlow `@task`
- 의존성 표기: `>>`, `<<`, `set_upstream`, `set_downstream`
- 설계 품질은 의존성/재시도/실패 전파 제어 품질로 드러남

In [ ]:
# === Checkpoint 2-1: 컴포넌트 관찰 ===
docker compose -f compose.yml logs --tail=80 airflow-scheduler

docker compose -f compose.yml logs --tail=80 airflow-dag-processor

docker compose -f compose.yml logs --tail=80 airflow-apiserver

---
## 3. Airflow UI 운영 동선 (Light Mode)

UI는 "실행/장애/재실행/원인 파악"의 중심 도구입니다.

### 3.1 Home
![Home (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/home_light.png)
- MetaDatabase/Scheduler/Triggerer/Dag Processor 상태 확인
- 상태별 DAG 바로가기(Failed/Running/Active)
- 장애 대응 시작점

### 3.2 Dag List
![Dag List (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_list1.png)
![Dag List Asset Condition Popup (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_list_asset_condition_popup1.png)
- 어떤 DAG가 실패/정지/실행 중인지 1차 분류
- 태그/상태/검색 필터로 운영 우선순위 정리

### 3.3 Dag Details
![Dag Overview Dashboard (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_dashboard1.png)
![Dag Grid View (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_grid1.png)
![Dag Graph View (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_graph1.png)
- Overview: 최근 실패/실행 요약
- Grid: 행=Task, 열=Dag Run
- Graph: 의존성 구조와 실행 상태 해석

### 3.4 Dag Tabs
![Dag Runs Tab (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_runs1.png)
![Dag Tasks Tab (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_tasks1.png)
![Dag Events Tab (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_events1.png)
![Dag Code Tab (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_code1.png)
![Dag Details Tab (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_overview_details1.png)
- Runs: Run 상태/시간/run type
- Tasks: operator type, trigger rule, 최근 상태
- Events: trigger/version patch 이벤트
- Code/Details: 코드 스냅샷/설정 및 버전 확인

### 3.5 Dag Run / Task Instance
![Dag Run Task Instances (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_run_task_instances1.png)
![Dag Run Code Snapshot airflow() (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_run_code_hello_airflow1.png)
![Dag Run Code Snapshot world() (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_run_code_hello_world1.png)
![Dag Run Details (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_run_details1.png)
![Dag Run Graph (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_run_graph1.png)
![Task Instance Logs (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_task_instance_logs1.png)
![Task Instance XCom (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_run_task_instance_xcom1.png)
![Task Instance Details (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_task_instance_details1.png)
- Dag Run은 실행 1건 단위 포렌식 단위
- Task 실패 시 기본 순서: `Logs -> Rendered Templates/XCom -> Details`

### 3.6 Dag Trigger Window
![Dag Trigger Window Single Run (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_trigger_window_single_run.png)
![Dag Trigger Window Backfill (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/backfill.png)
- Single Run: 즉시 1회 실행
- Backfill: 과거 구간 재처리
- 단건 검증과 기간 재처리는 목적이 다름

### 3.7 Assets
![Asset List/Graph (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/asset_list_consuming_dags1.png)
![Asset Graph View (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/asset_view1.png)
![Dag Graph All Dependencies (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_graph_all_dependencies1.png)
![Dag Graph External Conditions (Light)](https://airflow.apache.org/docs/apache-airflow/stable/_images/dag_graph_external_conditions1.png)
- 자산 생산자/소비자 DAG 관계 추적
- 이벤트 기반 오케스트레이션에서 의존성 가시성 강화

### 3.8 Admin
![Variables (Hidden Value)](https://airflow.apache.org/docs/apache-airflow/stable/_images/variable_hidden.png)
![Admin Connections](https://airflow.apache.org/docs/apache-airflow/stable/_images/admin_connections.png)
![Admin Connections Add](https://airflow.apache.org/docs/apache-airflow/stable/_images/admin_connections_add.png)
- Variables/Connections/Pools 등 운영 설정 관리
- Admin 화면은 RBAC 권한 사용자에게만 노출

In [ ]:
# === Checkpoint 3-1: UI 진입 전 CLI 확인 ===
docker compose -f compose.yml exec -T airflow-scheduler airflow dags list

docker compose -f compose.yml exec -T airflow-scheduler airflow dags unpause beginner_hello

docker compose -f compose.yml exec -T airflow-scheduler airflow dags trigger beginner_hello

### UI 체크포인트 (Section 3)

1. Home에서 컴포넌트 상태가 모두 healthy인지 확인
2. Dags에서 `beginner_hello`, `day29_dag_basics_demo` 검색
3. `beginner_hello` 상세 화면
- Grid에서 `print_date -> say_hello -> done` 순서를 확인
- Graph에서 직렬 의존성 확인
4. Dag Run 상세에서 Task Instance 로그 1개 열기

---
## 4. 운영자 관점 장애 분석 루프

권장 루프:
1. Home에서 실패 비율 상승 감지
2. Dag List에서 실패 DAG 선별
3. Dag Details Grid에서 반복 실패 Task 찾기
4. Task Instance `Logs/Rendered Templates/XCom/Details` 확인
5. 단건 검증은 Single Run, 기간 재처리는 Backfill로 분리

추가 운영 포인트:
- DAG Run 최종 상태는 leaf task 상태 규칙을 따름
- `catchup`, `backfill`, `clear`는 목적과 범위를 구분해 사용
- trigger rule(`all_done` 등) 오남용 시 "실패 숨김"이 생길 수 있음

In [ ]:
# === Checkpoint 4-1: 실행 상태/로그 확인 ===
docker compose -f compose.yml exec -T airflow-scheduler airflow dags list-runs -d beginner_hello

docker compose -f compose.yml exec -T airflow-scheduler airflow tasks list beginner_hello --tree

docker compose -f compose.yml logs --tail=80 airflow-scheduler

---
## 5. 정리

- Airflow는 오케스트레이션 플랫폼이다.
- 운영 속도는 UI 화면을 "Dag / Dag Run / Task Instance" 단위로 분리해서 볼 때 빨라진다.
- 배포 아키텍처를 분리할수록 보안/가용성은 좋아지지만 운영 설계 복잡도도 함께 증가한다.
- 초급 운영 루틴은 `Home -> Dag List -> Grid -> Task Logs` 순서로 고정하면 된다.

---


# ⚡ 2교시: DAG 기초 실습 + 운영 개념 확장

## 🎯 학습 목표
- DAG 선언 방식 3가지를 구분하고, 입문 단계 권장 패턴을 선택할 수 있다.
- DAG Run과 Task Instance의 관계, 상태 규칙을 설명할 수 있다.
- `>>`, Branching, Trigger Rule로 제어 흐름을 설계할 수 있다.
- Dynamic DAG, TaskGroup, doc_md, `.airflowignore` 등 심화 요소를 운영 관점에서 설명할 수 있다.
- Workloads(재실행/timeout/retry)와 Communication(XCom/Variables/Params) 핵심을 연결해 볼 수 있다.

---
## 1. DAG의 역할

DAG는 아래 4가지를 정의하는 설계도입니다.
- schedule: 언제 실행할지
- tasks: 어떤 작업 단위가 있는지
- dependencies: 어떤 순서/조건으로 실행할지
- callbacks/정책: 실패/성공 후 동작 규칙

기억할 점:
- DAG는 "데이터 계산 로직"보다 "실행 오케스트레이션"을 담당합니다.

![Basic DAG](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/basic_dag.png)

---
## 2. DAG 선언 방식 3가지

### 2.1 Context Manager (입문 권장)
```python
from datetime import datetime
from airflow.sdk import DAG
from airflow.providers.standard.operators.empty import EmptyOperator

with DAG(
    dag_id="sample_with_dag",
    start_date=datetime(2025, 1, 1),
    schedule="@daily",
    catchup=False,
):
    EmptyOperator(task_id="start")
```

### 2.2 Standard Constructor
```python
dag = DAG(dag_id="sample_constructor", start_date=datetime(2025, 1, 1), schedule="@daily")
EmptyOperator(task_id="start", dag=dag)
```

### 2.3 `@dag` 데코레이터
```python
from airflow.sdk import dag

@dag(start_date=datetime(2025, 1, 1), schedule="@daily", catchup=False)
def sample_decorator():
    EmptyOperator(task_id="start")

sample_decorator()
```

비교:
- `with DAG(...)`: 가장 읽기 쉽고 실무/입문 모두 기본 선택
- Standard Constructor: 레거시 코드에서 자주 만남
- `@dag`: TaskFlow와 함께 쓸 때 깔끔함

---
## 3. DAG 로딩과 Top-level 규칙

Airflow는 DAG 파일 import 후 **Top-level DAG 객체**를 수집합니다.

- Top-level에 선언된 DAG: 발견됨
- 함수 내부에서만 생성된 DAG: 발견되지 않을 수 있음

따라서 실습 파일은 아래를 지킵니다.
- DAG 객체를 모듈 최상단에서 생성
- `dags/` 하위에 저장
- 파일명과 `dag_id`를 명확히 구분

추가 포인트:
- DAG discovery safe mode가 켜져 있으면 파일 스캔 조건이 더 보수적일 수 있음
- import 시점에 무거운 I/O/외부 호출을 넣으면 파싱 지연의 원인이 됨

In [ ]:
# === Checkpoint 3-1: DAG 파일 인식 확인 ===
cd Day29-airflow-beginner-lab

docker compose -f compose.yml exec -T airflow-scheduler \
  sh -lc "airflow dags list 2>/dev/null | grep -E 'beginner_hello|day29_dag_basics_demo'"

docker compose -f compose.yml exec -T airflow-scheduler airflow dags list-import-errors

---
## 4. 의존성과 제어 흐름

### 4.1 `>>`, `<<` 연산자
- `a >> b`: a 성공 후 b 실행
- `a >> [b, c]`: fan-out
- `[b, c] >> d`: fan-in

### 4.2 `cross_downstream` / `chain`
```python
from airflow.sdk import cross_downstream, chain

cross_downstream([op1, op2], [op3, op4])
chain(op1, [op2, op3], [op4, op5], op6)
```

### 4.3 Branching
- 분기 함수가 반환한 Task ID만 실행
- 선택되지 않은 Task는 `skipped`

![Branch note](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/branch_note.png)

### 4.4 Trigger Rule
- 분기 후 join Task는 기본값(`all_success`) 대신
  `none_failed_min_one_success`나 `all_done`을 검토
- 실습 DAG는 `none_failed_min_one_success` 사용

![Branch without trigger rule](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/branch_without_trigger.png)
![Branch with trigger rule](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/branch_with_trigger.png)

### 4.5 LatestOnlyOperator / DependsOnPast
- LatestOnlyOperator: 최신 실행에서만 특정 downstream 실행
- depends_on_past=True: 이전 run의 같은 task 성공이 현재 실행 조건

![Latest Only with Trigger](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/latest_only_with_trigger.png)

In [ ]:
# === Checkpoint 4-1: DAG 단건 테스트 ===
docker compose -f compose.yml exec -T airflow-scheduler \
  airflow dags test day29_dag_basics_demo 2026-02-19

---
## 5. 실습 DAG: `day29_dag_basics_demo`

실습 DAG 구조:

`start -> prepare -> choose_branch -> [path_even | path_odd] -> join -> finish`

분기 기준:
- `dag_run.conf.sample_value`가 짝수면 `path_even`
- 홀수면 `path_odd`

join은 `none_failed_min_one_success`로 설정하여
스킵 전파 때문에 join이 막히지 않도록 구성합니다.

운영 관점 확인 포인트:
- branch 선택/스킵 상태를 Graph/Grid에서 동시에 검증
- join trigger rule이 의도대로 동작하는지 로그로 확인

In [ ]:
# === Checkpoint 5-1: 수동 트리거(짝수/홀수) ===
docker compose -f compose.yml exec -T airflow-scheduler airflow dags unpause day29_dag_basics_demo

docker compose -f compose.yml exec -T airflow-scheduler \
  airflow dags trigger day29_dag_basics_demo --conf '{"sample_value": 2}'

docker compose -f compose.yml exec -T airflow-scheduler \
  airflow dags trigger day29_dag_basics_demo --conf '{"sample_value": 3}'

### UI 체크포인트 (Section 5)

1. Graph View에서 분기 노드(`choose_branch`) 확인
2. 짝수 실행에서는 `path_even`만 성공, `path_odd`는 skipped 확인
3. 홀수 실행에서는 반대로 동작하는지 확인
4. join Task가 `none_failed_min_one_success`로 성공하는지 확인

In [ ]:
# === Checkpoint 5-2: 최근 Run 상태 확인 ===
docker compose -f compose.yml exec -T airflow-scheduler airflow dags list-runs -d day29_dag_basics_demo

---
## 6. DAG 심화 운영 포인트

### 6.1 Dynamic DAG
- 반복문으로 유사 task를 자동 생성
- `task_id` 고유성 필수, parse 시 무거운 로직 금지

### 6.2 TaskGroup / Edge Labels
![TaskGroup](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/task_group.gif)
![Edge Label example](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/edge_label_example.png)
- TaskGroup: UI 가독성 개선
- Edge Label: 분기 이유를 연결선에 문서화

### 6.3 문서화(doc_md)
- `dag.doc_md`, `task.doc_md`로 운영 컨텍스트를 UI에 남김
- 온보딩/장애 대응 시 맥락 전달 비용 감소

### 6.4 Packaging DAGs / `.airflowignore`
- DAG를 ZIP 패키지로 배포 가능(순수 Python만)
- `.airflowignore`로 실험 파일 파싱 제외

### 6.5 DAG Dependencies
- TriggerDagRunOperator: 능동 트리거
- ExternalTaskSensor: 수동 대기

### 6.6 DAG 생명주기 / Auto-pausing
- Pause: 자동 스케줄 중단, 수동 실행 가능
- Deactivate: 파일 제거 기반 비활성화(이력 보존)
- Delete: 메타데이터 삭제(이력 소실)
- `max_consecutive_failed_dag_runs`로 연속 실패 자동 정지

---
## 7. Workloads 실행/운영 제어 포인트 (05 내용 반영)

### 7.1 DAG Run 상태 규칙
- DAG Run 최종 상태는 leaf task 기준
- `success`: 모든 leaf가 success/skipped
- `failed`: leaf에 failed/upstream_failed 존재

### 7.2 Data Interval / Logical Date / Runtime
- data interval: 처리 대상 시간 구간
- logical date: interval 시작 기준 시각
- runtime: 실제 실행 시각

### 7.3 Re-run 전략
- catchup: 누락 구간 자동 생성
- backfill: 특정 과거 구간 명시 재처리
- clear: 특정 Task Instance 범위 재실행

![Task instance history](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/task_instance_history.png)
![Task instance history log](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/task_instance_history_log.png)

### 7.4 Task lifecycle / Timeouts / Heartbeat
![Task lifecycle](https://airflow.apache.org/docs/apache-airflow/3.0.4/_images/diagram_task_lifecycle.png)
- `execution_timeout` vs sensor `timeout` 구분
- heartbeat timeout으로 좀비 task 정리
- executor_config로 task 단위 실행정책 분리

### 7.5 Special Exceptions / SLA 참고
- `AirflowSkipException`, `AirflowFailException`으로 상태 제어
- Airflow 3.0 기준 SLA 동작 변경 사항 확인 필요(버전별 문서 확인)

---
## 8. Task 작성/Communication 연결 포인트 (06~07 내용 요약)

### 8.1 Task 작성
- Operator: 재사용 템플릿
- Sensor: 대기 작업(`poke`/`reschedule`)
- TaskFlow `@task`: Python 함수 기반 작성
- 템플릿/Jinja, `params` 예약 키워드, native object 렌더링 주의

### 8.2 Communication 3축
- XCom: run 내부 task 간 소량 전달
- Variables: 인스턴스 전역 설정값
- Params: DAG Run 입력 + JSON Schema 검증

### 8.3 운영 실무 규칙
- 대용량 데이터 전달은 XCom 대신 외부 스토리지 사용
- 런타임 입력 품질은 Params 스키마로 강제
- 설정 관리는 Variables 남용보다 코드/리포지토리 우선

---
## 9. 정리

- 입문 단계 기본 패턴은 `with DAG(...)` + `>>` 입니다.
- DAG 발견 문제는 Top-level 규칙으로 대부분 해결됩니다.
- 분기 이후 join은 Trigger Rule을 의식해서 설계해야 합니다.
- 운영 품질은 DAG 구조(TaskGroup/doc) + 실행 해석(DAG Run/TI) + 제어 정책(timeout/retry)에서 결정됩니다.